In [ ]:
#BUSINESS DATA FILE LINK

import pandas as pd

# 1.link the Excel file
file = '/Users/rajhomedesktop/Desktop/TimberLens-Creations LLC/TimberLens-Creations_Business_Plan_FY2026.xlsx'
xl = pd.ExcelFile(file)

# 2. Establish the context for the agents
full_plan_context = ""
for sheet in xl.sheet_names:
    df_sheet = pd.read_excel(file, sheet_name=sheet)
    full_plan_context += f"\n--- {sheet} ---\n{df_sheet.to_string()}\n"

print("✅ 'xl' is defined and Business Plan context is loaded.")

In [ ]:
#SYSTEM CONFIGURATOR 

import os
import anthropic
import pandas as pd
import re

# --- 1. THE PATH DEFINITIONS ---
NEW_API_PATH = '/Users/rajhomedesktop/Desktop/TimberLens-Creations LLC/CLAUDE_API_KEY.rtf'
BASE_PATH = "/Users/rajhomedesktop/anaconda_projects/Anthropic AI"
SKILLS_PATH = os.path.join(BASE_PATH, "Skills")
EXCEL_FILE_PATH = '/Users/rajhomedesktop/Desktop/TimberLens-Creations LLC/TimberLens-Creations_Business_Plan_FY2026.xlsx'

# --- 2. THE API KEY CHECK (.rtf extraction) ---
def get_clean_key(path):
    try:
        with open(path, 'r') as f:
            content = f.read()
            # Extracts the sk-ant- key from potential RTF formatting
            match = re.search(r'sk-ant-[\w-]+', content)
            return match.group(0) if match else None
    except:
        return None

api_key = get_clean_key(NEW_API_PATH)

if api_key:
    os.environ["ANTHROPIC_API_KEY"] = api_key
    client = anthropic.Anthropic(api_key=api_key)
    masked = f"{api_key[:7]}****************{api_key[-4:]}"
    print(f"✅ API: Key loaded and masked: {masked}")
else:
    print(f"❌ API: Could not find key in {NEW_API_PATH}")

# --- 3. SKILLS_PATH CHECK ---
if os.path.exists(SKILLS_PATH):
    skills_count = len([f for f in os.listdir(SKILLS_PATH) if f.endswith('.md')])
    print(f"✅ SKILLS_PATH: Found {skills_count} skills at {SKILLS_PATH}")
else:
    print(f"❌ SKILLS_PATH: Directory not found at {SKILLS_PATH}")

# --- 4. EXCEL CHECK ---
try:
    xl_check = pd.ExcelFile(EXCEL_FILE_PATH)
    print(f"✅ Excel: Linked successfully with sheets: {xl_check.sheet_names}")
except Exception as e:
    print(f"❌ Excel: Connection failed at {EXCEL_FILE_PATH}. Error: {e}")

# --- 5. INITIALIZATION COMPLETE ---
print("\n🚀 System ready for Orchestrator!!")

In [ ]:
#E-MAIL & PDF GENERATOR 

import smtplib
from email.message import EmailMessage
import markdown
import re
import os
from fpdf import FPDF

def get_credentials():
    """Extracts email and app password from your RTF file."""
    # Using the path already established above
    RTF_PATH = '/Users/rajhomedesktop/Desktop/TimberLens-Creations LLC/CLAUDE_API_KEY.rtf'
    with open(RTF_PATH, 'r') as f:
        content = f.read()
        email = re.search(r'[\w\.-]+@[\w\.-]+', content).group(0)
        # Finds the 16-char Google App Password
        password = re.search(r'[a-z]{4}\s[a-z]{4}\s[a-z]{4}\s[a-z]{4}', content).group(0)
    return email, password

def send_approval_email(subject, body_text, attachment_path=None):
    """The Email Engine."""
    user_email, app_password = get_credentials()
    msg = EmailMessage()
    msg['Subject'] = subject
    msg['From'] = user_email
    msg['To'] = user_email
    
    # Converts Manager output to professional HTML tables
    html_body = markdown.markdown(body_text, extensions=['tables'])
    msg.set_content(body_text)
    msg.add_alternative(html_body, subtype='html')

    if attachment_path and os.path.exists(attachment_path):
        with open(attachment_path, 'rb') as f:
            file_data = f.read()
            msg.add_attachment(file_data, maintype='application', subtype='pdf', 
                               filename=os.path.basename(attachment_path))

    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login(user_email, app_password)
        smtp.send_message(msg)

def trigger_dispatch_if_requested(user_prompt, report_text):
    """The Decision Logic."""
    trigger_words = ["email", "pdf", "send"]
    if any(word in user_prompt.lower() for word in trigger_words):
        print("📨 Request detected! Preparing PDF and Email dispatch...")
        subject = "TimberLens Request: Invoice Draft"
        
        # PDF Creation
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", size=10)
        clean_text = report_text.encode('ascii', 'ignore').decode('ascii').replace('#', '').replace('*', '')
        pdf.multi_cell(0, 8, text=clean_text)
        
        pdf_path = os.path.join(os.path.expanduser("~"), "Desktop/TimberLens-Creations LLC/Requested_Report.pdf")
        pdf.output(pdf_path)
        
        try:
            send_approval_email(subject, report_text, attachment_path=pdf_path)
            print(f"✅ Dispatch Successful: PDF saved to Desktop and Sent to Email.")
        except Exception as e:
            print(f"⚠️ Email failed but PDF saved: {e}")
    else:
        print("ℹ️ No email/pdf requested. Staying in-notebook only.")

In [ ]:
#MAIN ORCHESTRATOR ENGINE 

import os
import pandas as pd
from IPython.display import display, Markdown  # Added for visual formatting

# --- 1. THE FIXED CONFIGURATION ---
MODEL_NAME = "claude-opus-4-6" 

def sync_excel_to_state():
    """Reads Excel and overwrites System_State.md with fresh data."""
    print("🔄 Syncing Excel data to System State...")
    try:
        xl = pd.ExcelFile(EXCEL_FILE_PATH)
        full_plan_context = f"# TIMBERLENS LIVE STATE\nLast Sync: {pd.Timestamp.now()}\n"
        for sheet in xl.sheet_names:
            df_sheet = pd.read_excel(EXCEL_FILE_PATH, sheet_name=sheet)
            full_plan_context += f"\n--- {sheet} ---\n{df_sheet.to_string()}\n"
        
        state_path = os.path.join(SKILLS_PATH, "System_State.md")
        with open(state_path, "w") as f:
            f.write(full_plan_context)
        print("✅ Sync Complete.")
    except Exception as e:
        print(f"⚠️ Sync Failed: {e}")

def ask_skill(skill_name, user_input, extra_context=""):
    """Queries a specific .md skill file while injecting the Live Excel State."""
    skill_path = os.path.join(SKILLS_PATH, f"{skill_name}.md")
    state_path = os.path.join(SKILLS_PATH, "System_State.md")
    
    with open(skill_path, "r") as f:
        skill_prompt = f.read()
    with open(state_path, "r") as f:
        system_state = f.read()
        
    combined_system = f"{skill_prompt}\n\n### LIVE BUSINESS DATA ###\n{system_state}\n{extra_context}"
    
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4000,
        system=combined_system,
        messages=[{"role": "user", "content": user_input}]
    )
    return response.content[0].text

def timberlens_orchestrator(user_prompt):
    """The permanent brain of the app. Routes and renders beautifully."""
    sync_excel_to_state()
    
    # 1. Team Discussion
    team = ["Master Artisan", "SupplyChain", "Finance", "WebDev", "Marketing", "Inventory"]
    roll_call_results = ""
    
    for member in team:
        print(f"📡 Pinging {member}...")
        response = ask_skill(member, user_prompt)
        roll_call_results += f"\n[{member.upper()}]: {response}\n"
        
    # 2. Final Synthesis with Visual Instructions
    visual_instructions = """
    \n\n### VISUAL STANDARDS FOR SECTION 1:
    - Table MUST have exactly 4 columns: [Cost Category | Walnut | Cherry | % of Total]
    - Do NOT use '$' or symbols inside the numerical cells (keep them in headers).
    - Use bold text for the Category name only.
    - Ensure the 'TOTAL' row is at the bottom.
    - If a value is identical for both, still list it in both columns.
    """

    # We also update this to allow for more tokens (4096) to prevent the cutoff
    final_raw_text = ask_skill(
        "Manager", 
        f"The team has responded to: '{user_prompt}'. Summarize as an Executive Report.", 
        extra_context=roll_call_results + visual_instructions
    )
    # NEW: The Automation Bridge
    trigger_dispatch_if_requested(user_prompt, final_raw_text)
    
    # # 3. VISUAL RENDER
    return display(Markdown(final_raw_text))

In [ ]:
#CONTEXT HUB BRIDGE

import subprocess

def fetch_hub_docs(doc_id):
    """Fetches clean docs from the Hub without cluttering the notebook."""
    try:
        result = subprocess.run(['chub', 'get', doc_id, '--lang', 'py'], 
                                capture_output=True, text=True)
        return result.stdout if result.returncode == 0 else "Doc not found."
    except FileNotFoundError:
        return "Chub CLI not installed. Run API_Setup.ipynb first."

print("🛠️ Context Hub Bridge active. Ready to fetch documentation.")

In [ ]:
# 🪵 TIMBERLENS COMMAND CONSOLE

user_request = "do a competitive analysis of guitar stands in my area, send me email with pdf"
final_raw_text = timberlens_orchestrator(user_request)